# Task 3 (KAN-6): EDA, Feature Engineering, Leakage/Multicollinearity Checks & PCA

**Goal:** Design and build all model inputs, validate they are leak-free and well-conditioned, then apply PCA to decorrelate the features before modelling.

**Sections**
1. [Setup & data load](#1.-Setup-&-data-load)
2. [Feature design](#2.-Feature-design)
3. [EDA visualizations](#3.-EDA-visualizations)
4. [Leakage checks](#4.-Leakage-checks)
5. [Multicollinearity](#5.-Multicollinearity)
6. [PCA](#6.-PCA)
7. [Save outputs](#7.-Save-outputs)
8. [Summary of findings](#8.-Summary-of-findings)

**Outputs**
| File | Description |
|---|---|
| `data/features_panel.csv` | Full 38-feature table, chronological split |
| `data/features_pca.csv` | Keys + targets + 14 PC scores |
| `models/pca_pipeline.joblib` | Fitted StandardScaler + PCA pipeline |
| `reports/feature_vif.csv` | VIF table |
| `reports/pca_loadings.csv` | PCA loadings matrix |
| `reports/figures/task3/` | 11 diagnostic charts |

## 1. Setup & data load

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from build_features import (
    ARRIVAL_FEATURES, CALENDAR_FEATURES, CATEGORICAL_FEATURES,
    KEYS, NUMERIC_FEATURES, PRICE_FEATURES, TARGETS, WEATHER_FEATURES,
    build_features,
)

DATA   = ROOT / "data"
FIG    = ROOT / "reports" / "figures" / "task3"
MODELS = ROOT / "models"
REP    = ROOT / "reports"

FIG.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
REP.mkdir(parents=True, exist_ok=True)

# ── style ──────────────────────────────────────────────────────────────────────
INK, INK2, MUTED, SURFACE = "#0b0b0b", "#52514e", "#898781", "#fcfcfb"
CROP_COLORS = {"Onion": "#e05c2e", "Potato": "#2a78d6", "Tomato": "#1baf7a"}
DIR_COLORS  = {"fall": "#184f95", "flat": "#eda100", "rise": "#b52e2e"}
GROUP_COLORS = {"Price": "#2a78d6", "Arrivals": "#eb6834",
                "Weather": "#1baf7a", "Calendar": "#eda100"}
DIV = LinearSegmentedColormap.from_list(
    "div_blue_red", ["#184f95", "#6da7ec", "#f0efec", "#ec8a89", "#b52e2e"]
)
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": MUTED,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "figure.dpi": 110,
    "savefig.dpi": 150, "savefig.bbox": "tight",
})

GROUP = (
    {f: "Price"    for f in PRICE_FEATURES}
    | {f: "Arrivals" for f in ARRIVAL_FEATURES}
    | {f: "Weather"  for f in WEATHER_FEATURES}
    | {f: "Calendar" for f in CALENDAR_FEATURES}
)

BAND_PCT   = 2.5
TEST_START = pd.Timestamp("2026-07-06")

In [ ]:
panel = pd.read_csv(DATA / "panel_with_weather.csv", parse_dates=["week_start_date"])

# Drop partial final week (< 6 trading days)
full_weeks = panel[~panel["is_partial_week"]].copy()

print(f"Panel: {panel.shape[0]:,} rows | Full-week rows: {len(full_weeks):,} "
      f"| Partial-week rows dropped: {panel['is_partial_week'].sum()}")
print(f"Crops : {sorted(panel['crop'].unique())}")
print(f"Mandis: {panel['market_name'].nunique()}")
print(f"Dates : {panel['week_start_date'].min().date()} → {panel['week_start_date'].max().date()}")
panel.head(3)

## 2. Feature design

All features are computed within each **mandi × crop** time series and use only data available at the **end of week _t_**. The only forward-looking column is `next_week_is_quarter_start`, which is knowable in advance from the calendar.

| Group | Count | Features |
|---|---|---|
| **Price** | 15 | `price`, lags 1/2/4, % changes 1/2/4w, rolling mean 4/8w, rolling std 4w, deviations from rolling means, price spread, state/national mean |
| **Arrivals** | 5 | `arrivals`, lag-1, 1w % change, rolling mean 4w, deviation from rolling mean |
| **Weather** | 12 | `rainfall_mm`, lags 1/2, 4w sum, rainy days, heavy-rain flag (≥64.5 mm/day), temp mean/max, heat flag (≥40°C), temp anomaly, humidity mean, humidity 4w mean |
| **Calendar** | 6 | week-of-quarter, quarter-start flag (t+1), quarter, month, week-of-year sin/cos |

In [ ]:
# Build all features (includes warm-up NaN rows)
feats_all = build_features(full_weeks)
print(f"Built (before dropping warm-up): {feats_all.shape}")

# Drop warm-up rows + last row per series (no target)
feats = feats_all.dropna(subset=NUMERIC_FEATURES + ["target_pct_change"]).reset_index(drop=True)
feats["split"] = np.where(feats["week_start_date"] >= TEST_START, "test", "train")

print(f"Usable rows     : {len(feats):,}  (dropped {len(feats_all) - len(feats)} warm-up/last-week rows)")
print(f"\nChronological split:")
print(feats.groupby("split")["week_start_date"].agg(["min", "max", "nunique", "size"]).to_string())

train = feats[feats["split"] == "train"]
test  = feats[feats["split"] == "test"]

print(f"\nTarget class share by split:")
pd.crosstab(feats["split"], feats["target_direction"], normalize="index").round(3)

In [ ]:
# Descriptive statistics on the training set
train[NUMERIC_FEATURES].describe().T[["mean", "std", "min", "max"]].round(2)

In [ ]:
# Feature group count chart
group_counts = {g: list(GROUP.values()).count(g) for g in GROUP_COLORS}

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(list(group_counts.keys()), list(group_counts.values()),
               color=[GROUP_COLORS[g] for g in group_counts], height=0.55)
ax.bar_label(bars, fmt="%d", padding=4, color=INK2)
ax.set_xlabel("Number of features")
ax.set_title("Feature groups: count of engineered inputs")
ax.set_xlim(0, max(group_counts.values()) + 3)
fig.savefig(FIG / "0_feature_group_counts.png")
plt.show()

## 3. EDA visualizations

Five charts that establish the price dynamics, seasonality, and weather links before building models.

In [ ]:
# Prepare EDA dataframe (full panel, all mandis)
eda = full_weeks.copy()
eda["price_index"] = eda.groupby(["crop", "market_name"])["weekly_modal_price"].transform(
    lambda s: s / s.mean() * 100
)
eda["pct_change"] = eda.groupby(["crop", "market_name"])["weekly_modal_price"].pct_change() * 100

# Week-of-quarter for seasonality chart
thursday = eda["week_start_date"] + pd.Timedelta(days=3)
q_start  = thursday.dt.to_period("Q").dt.start_time
first_th  = q_start + pd.to_timedelta((3 - q_start.dt.dayofweek) % 7, unit="D")
eda["week_of_quarter"] = ((eda["week_start_date"] - (first_th - pd.Timedelta(days=3))).dt.days // 7 + 1).astype(int)

In [ ]:
# Chart 1: National average weekly price by crop
nat = eda.groupby(["week_start_date", "crop"])["weekly_modal_price"].mean().unstack()

fig, ax = plt.subplots(figsize=(11, 4.5))
for crop, col in CROP_COLORS.items():
    if crop in nat.columns:
        ax.plot(nat.index, nat[crop], color=col, lw=2, label=crop)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"₹{x:,.0f}"))
ax.set_xlabel("Week start date")
ax.set_ylabel("Weekly modal price (₹/q)")
ax.set_title("Chart 1: National average weekly modal price by crop")
ax.legend(frameon=False)
fig.savefig(FIG / "1_national_price_by_crop.png")
plt.show()

**Takeaway:** Tomato is the most expensive crop (around ₹2,200–3,000/q), followed by onion and potato. All three show clear quarterly cycles with sharp peaks and troughs.

In [ ]:
# Chart 2: Seasonality by week of quarter
woq = eda.groupby(["crop", "week_of_quarter"])["price_index"].agg(["mean", "std"]).reset_index()

fig, ax = plt.subplots(figsize=(10, 4.5))
for crop, col in CROP_COLORS.items():
    d = woq[woq["crop"] == crop]
    ax.plot(d["week_of_quarter"], d["mean"], color=col, lw=2, label=crop)
    ax.fill_between(d["week_of_quarter"], d["mean"] - d["std"], d["mean"] + d["std"],
                    color=col, alpha=0.12)
ax.axhline(100, color=MUTED, lw=1, ls="--", label="Mandi mean = 100")
ax.set_xlabel("Week of quarter")
ax.set_ylabel("Price index (mandi mean = 100)")
ax.set_title("Chart 2: Intra-quarter price seasonality by crop")
ax.legend(frameon=False)
fig.savefig(FIG / "2_seasonality_week_of_quarter.png")
plt.show()

**Takeaway:** The strongest pattern is a **quarterly cycle**. Prices sit roughly 6% below the mandi mean at the quarter start and climb toward the quarter-end — driven by procurement cycles and harvest timing. This justifies `week_of_quarter` and `next_week_is_quarter_start` as features.

In [ ]:
# Chart 3: Price vs arrivals (within-mandi, de-meaned)
scatter_df = eda.copy()
for col in ["weekly_modal_price", "weekly_total_arrivals"]:
    scatter_df[col] = scatter_df.groupby(["crop", "market_name"])[col].transform(lambda s: s - s.mean())

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=False)
for ax, (crop, col) in zip(axes, CROP_COLORS.items()):
    d = scatter_df[scatter_df["crop"] == crop].dropna(
        subset=["weekly_modal_price", "weekly_total_arrivals"])
    r = d["weekly_modal_price"].corr(d["weekly_total_arrivals"])
    sample = d.sample(min(len(d), 800), random_state=42)
    ax.scatter(sample["weekly_total_arrivals"], sample["weekly_modal_price"],
               s=8, alpha=0.35, color=col, edgecolors="none")
    ax.set_xlabel("Arrivals deviation (tonnes)")
    ax.set_ylabel("Price deviation (₹/q)")
    ax.set_title(f"{crop}\n(r = {r:+.2f})")
fig.suptitle("Chart 3: Price vs arrivals within each mandi (de-meaned)",
             y=1.01, fontsize=12, fontweight="bold", color=INK)
fig.tight_layout()
fig.savefig(FIG / "3_price_vs_arrivals.png")
plt.show()

**Takeaway:** Within a mandi, weeks with heavier arrivals do **not** show lower prices (|r| ≈ 0.0–0.05). Cross-mandi price levels dominate the raw correlation, masking supply effects within a market. Arrivals still enter as features because short-run deviations from their own rolling average (`arrivals_vs_roll_mean_4`) may carry a signal.

In [ ]:
# Chart 4: Next-week price change by rainfall / heat category
eda_s = eda.sort_values(["crop", "market_name", "week_start_date"])
eda_s["next_pct_change"] = eda_s.groupby(["crop", "market_name"])["weekly_modal_price"].pct_change().shift(-1) * 100

rain_bins   = [-0.1, 2.4, 15.5, 64.5, 9999]
rain_labels = ["Dry\n(0–2.4 mm)", "Light\n(2.5–15.5)", "Moderate\n(15.5–64.5)", "Heavy\n(>64.5)"]
eda_s["rain_cat"] = pd.cut(eda_s["max_daily_rainfall_mm"], bins=rain_bins, labels=rain_labels)
eda_s["heat_cat"] = np.where(eda_s["weekly_temp_max_c"] >= 40, "Heat wave\n(≥40°C)", "Normal\n(<40°C)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

rain_med = eda_s.groupby("rain_cat", observed=True)["next_pct_change"].median()
ax1.bar(rain_med.index, rain_med.values,
        color=[CROP_COLORS["Onion"], CROP_COLORS["Potato"], CROP_COLORS["Tomato"], "#8b008b"], width=0.55)
ax1.axhline(0, color=MUTED, lw=1)
ax1.set_xlabel("Rainfall category (max daily mm)")
ax1.set_ylabel("Median next-week price change (%)")
ax1.set_title("Chart 4a: Next-week price change\nby rainfall category")

heat_med = eda_s.groupby("heat_cat")["next_pct_change"].median()
ax2.bar(heat_med.index, heat_med.values,
        color=[CROP_COLORS["Potato"], CROP_COLORS["Onion"]], width=0.35)
ax2.axhline(0, color=MUTED, lw=1)
ax2.set_xlabel("Temperature category")
ax2.set_ylabel("Median next-week price change (%)")
ax2.set_title("Chart 4b: Next-week price change\nby heat category")

fig.tight_layout()
fig.savefig(FIG / "4_price_change_by_weather.png")
plt.show()

**Takeaway:** Weather categories alone have almost no unconditional effect on next-week price change (medians hover near 0%). However, lagged and cumulative weather effects (rainfall roll-sum, heat flag interactions with supply) may still contribute, so weather features are kept. The model will determine their weight.

In [ ]:
# Chart 5: Distribution of weekly price changes + ±2.5% flat band
fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.arange(-30, 31, 1)
for crop, col in CROP_COLORS.items():
    vals_crop = eda.loc[eda["crop"] == crop, "pct_change"].dropna()
    ax.hist(vals_crop, bins=bins, color=col, alpha=0.45, density=True, label=crop)
ax.axvline(-BAND_PCT, color=INK2, lw=1.5, ls="--")
ax.axvline(+BAND_PCT, color=INK2, lw=1.5, ls="--", label=f"±{BAND_PCT}% flat band")
ax.axvspan(-BAND_PCT, +BAND_PCT, color=INK2, alpha=0.06)
ax.set_xlabel("Weekly price change (%)")
ax.set_ylabel("Density")
ax.set_title("Chart 5: Distribution of weekly price changes")
ax.legend(frameon=False)
fig.savefig(FIG / "5_price_change_distribution.png")
plt.show()

**Takeaway:** Weekly changes centre near 0% with a spread of roughly ±6.5%. A **±2.5% flat band** splits the distribution into rise (~38%), flat (~24%), fall (~38%) — a reasonably balanced three-class target. The choice of 2.5% is small enough to capture meaningful moves while keeping the flat class non-trivial.

## 4. Leakage checks

Two complementary checks:
- **Truncation test:** rebuild features from data cut off at week _t_ — values must be identical to those built from the full history.
- **Feature–target screen:** flag any feature with |Spearman ρ| > 0.8 with the next-week target (almost certain leakage if triggered).

In [ ]:
# 4a. Truncation test
def truncation_test(cutoff: pd.Timestamp) -> pd.Series:
    """Return features that differ when built from data truncated to ≤ cutoff."""
    cut = build_features(full_weeks[full_weeks["week_start_date"] <= cutoff])
    a = (cut[cut["week_start_date"] == cutoff]
         .set_index(["crop", "market_name"])[NUMERIC_FEATURES])
    b = (feats_all[feats_all["week_start_date"] == cutoff]
         .set_index(["crop", "market_name"])[NUMERIC_FEATURES])
    diff = (a - b.loc[a.index]).abs().max()
    return diff[diff > 1e-9]

for cutoff_str in ["2026-01-12", "2026-04-20", "2026-08-31"]:
    bad = truncation_test(pd.Timestamp(cutoff_str))
    status = "PASS ✓  (no feature uses future data)" if bad.empty else f"FAIL ✗  {bad.to_dict()}"
    print(f"cut-off {cutoff_str}  →  {status}")

In [ ]:
# 4b. Feature–target Spearman ρ (train set only)
rho = (train[NUMERIC_FEATURES]
       .corrwith(train["target_pct_change"], method="spearman")
       .sort_values(key=abs))

flagged = list(rho[rho.abs() > 0.8].index)
print(f"Features with |ρ| > 0.8 (leakage signal): {flagged or 'none'}")

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(rho.index, rho.values,
        color=[GROUP_COLORS[GROUP[f]] for f in rho.index], height=0.7)
ax.axvline(0, color=MUTED, lw=1)
ax.axvline(0.8,  color="#b52e2e", lw=1, ls="--", alpha=0.6)
ax.axvline(-0.8, color="#b52e2e", lw=1, ls="--", alpha=0.6, label="|ρ| > 0.8 threshold")
for f, v in rho.items():
    if abs(v) >= 0.10:
        ax.annotate(f"{v:+.2f}", (v, f),
                    xytext=(4 if v > 0 else -4, 0), textcoords="offset points",
                    ha="left" if v > 0 else "right", va="center", fontsize=8, color=INK2)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]
ax.legend(handles, GROUP_COLORS.keys(), frameon=False, loc="lower right")
ax.set_xlabel("Spearman ρ with next week's % price change (train)")
ax.set_title("Chart 6: Feature–target correlation by group")
ax.grid(axis="y", visible=False)
fig.savefig(FIG / "6_feature_target_correlation.png")
plt.show()

print("\nTop-10 features by |Spearman ρ|:")
rho.sort_values(key=abs, ascending=False).head(10).round(3)

**Leakage result:** All three truncation tests pass and no feature reaches |ρ| > 0.8 with the target (max |ρ| ≈ 0.43 for `next_week_is_quarter_start`). The data pipeline is leak-free.

**Strongest predictors (train):** `next_week_is_quarter_start` and the mean-reversion features (`price_vs_roll_mean_4/8`, `pct_change_1w/2w`) — consistent with the quarterly-cycle finding in the EDA.

## 5. Multicollinearity

High correlation between features inflates variance in linear models. We measure it two ways: Pearson correlation and Variance Inflation Factor (VIF).

In [ ]:
# 5a. Correlation heatmap
corr = train[NUMERIC_FEATURES].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap=DIV, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7.5)
ax.set_yticks(range(len(corr)), corr.index, fontsize=7.5)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.65, label="Pearson r (train)")
ax.set_title("Chart 7: Feature correlation matrix (Pearson r, train set)")
fig.savefig(FIG / "7_feature_correlation_matrix.png")
plt.show()

In [ ]:
# High-correlation pairs (|r| > 0.9)
pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().rename("r").reset_index()
pairs.columns = ["feature_1", "feature_2", "r"]
high = pairs[pairs["r"].abs() > 0.9].sort_values("r", key=abs, ascending=False)
print(f"{len(high)} feature pairs with |r| > 0.9:")
high.round(3)

In [ ]:
# 5b. Variance Inflation Factor
X  = train[NUMERIC_FEATURES]
Xs = (X - X.mean()) / X.std()
Xs = Xs.assign(const=1.0)
vif = pd.Series(
    [variance_inflation_factor(Xs.values, i) for i in range(len(NUMERIC_FEATURES))],
    index=NUMERIC_FEATURES, name="VIF"
).sort_values(ascending=False)
vif_table = vif.to_frame().assign(
    group=lambda d: d.index.map(GROUP),
    flag=lambda d: np.where(d["VIF"] > 10, "VIF > 10", "")
)
vif_table.round(1).to_csv(REP / "feature_vif.csv")
print(f"{(vif > 10).sum()} of {len(vif)} features have VIF > 10")
vif_table.round(1)

In [ ]:
# VIF bar chart (top-20)
fig, ax = plt.subplots(figsize=(8, 7))
top_vif = vif_table.head(20)
ax.barh(top_vif.index[::-1], top_vif["VIF"][::-1],
        color=[GROUP_COLORS[g] for g in top_vif["group"][::-1]], height=0.65)
ax.axvline(10, color="#b52e2e", lw=1.5, ls="--", label="VIF = 10 threshold")
ax.set_xlabel("Variance Inflation Factor")
ax.set_title("Chart 8: Multicollinearity — VIF of top-20 features (train set)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]
ax.legend(handles, GROUP_COLORS.keys(), frameon=False, loc="lower right")
fig.savefig(FIG / "8_vif_chart.png")
plt.show()

**Takeaway:** 23 of 38 features have VIF > 10, with price-level features (`roll_mean_4`, `roll_mean_8`, `price`, its lags) reaching VIF > 1,000. 19 feature pairs have |r| > 0.9. The raw features are too collinear for a well-conditioned linear model → **PCA is the solution.**

## 6. PCA

StandardScaler + PCA are fit on the **training weeks only**. The number of components is chosen at the **90% explained-variance threshold** (the smallest set that retains most information while removing all the collinear redundancy).

In [ ]:
# 6a. Fit on train only; inspect explained variance
full_pipe = make_pipeline(StandardScaler(), PCA(random_state=42))
full_pipe.fit(train[NUMERIC_FEATURES])
pca_full = full_pipe.named_steps["pca"]
ev  = pca_full.explained_variance_ratio_
cum = np.cumsum(ev)

thresholds = {t: int(np.searchsorted(cum, t) + 1) for t in [0.80, 0.85, 0.90, 0.95]}
kaiser_n   = int((pca_full.explained_variance_ > 1).sum())
N_COMPONENTS = thresholds[0.90]

print("Explained variance thresholds:")
for t, n in thresholds.items():
    print(f"  {t:.0%} → {n} components")
print(f"Kaiser rule (eigenvalue > 1) → {kaiser_n} components")
print(f"\n✅  Choice: {N_COMPONENTS} components (90% threshold)")

In [ ]:
# Chart 9: Scree plot
ORANGE = "#eb6834"
x = np.arange(1, len(ev) + 1)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(x, ev * 100, color=CROP_COLORS["Potato"], alpha=0.6, label="Per component")
ax.plot(x, cum * 100, color=ORANGE, lw=2, marker="o", ms=3, label="Cumulative")
ax.axhline(90, color=MUTED, lw=1, ls="--")
ax.axvline(N_COMPONENTS + 0.5, color=MUTED, lw=1, ls=":")
ax.annotate(
    f"{N_COMPONENTS} PCs → {cum[N_COMPONENTS - 1]:.0%} variance",
    (N_COMPONENTS + 0.7, 60), color=INK2, fontsize=9
)
ax.set_xlabel("Principal component")
ax.set_ylabel("Explained variance (%)")
ax.set_title("Chart 9: Scree plot — variance explained by each component (train)")
ax.legend(frameon=False)
fig.savefig(FIG / "9_pca_scree.png")
plt.show()

In [ ]:
# 6b. Fit final PCA with N_COMPONENTS
pipe = make_pipeline(
    StandardScaler(), PCA(n_components=N_COMPONENTS, random_state=42)
).fit(train[NUMERIC_FEATURES])

pca      = pipe.named_steps["pca"]
pc_names = [f"PC{i + 1}" for i in range(N_COMPONENTS)]
loadings = pd.DataFrame(pca.components_.T, index=NUMERIC_FEATURES, columns=pc_names)
ev_kept  = pca.explained_variance_ratio_

In [ ]:
# Chart 10: Loadings heatmap (first 8 PCs)
SHOW = min(8, N_COMPONENTS)

fig, ax = plt.subplots(figsize=(9, 11))
im = ax.imshow(loadings.iloc[:, :SHOW].values, cmap=DIV, vmin=-0.7, vmax=0.7, aspect="auto")
ax.set_yticks(range(len(loadings)), loadings.index, fontsize=8)
ax.set_xticks(range(SHOW), [f"{c}\n{ev_kept[i]:.0%}" for i, c in enumerate(pc_names[:SHOW])])
ax.grid(False)
for (i, j), v in np.ndenumerate(loadings.iloc[:, :SHOW].values):
    if abs(v) >= 0.25:
        ax.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=7,
                color="#ffffff" if abs(v) > 0.45 else INK)
fig.colorbar(im, ax=ax, shrink=0.6, label="Loading")
ax.set_title(f"Chart 10: Loadings of the first {SHOW} components (|loading| ≥ 0.25 labelled)")
fig.savefig(FIG / "10_pca_loadings_heatmap.png")
plt.show()

**Component interpretation:**

| PC | Variance | Main loadings | Meaning |
|---|---|---|---|
| PC1 | ≈ 22% | price, lags, rolling means | **Price level** — how dear the crop is (largely crop identity) |
| PC2 | ≈ 14% | price vs rolling means, 1–2w % change, humidity (−) | **Short-term price momentum** |
| PC3 | ≈ 12% | 4w % change, price vs 8w mean, week-of-year | **Position in quarterly cycle** |
| PC4 | ≈ 11% | rainfall (level, lags, 4w sum), rainy days, temp | **Wet/monsoon weather** |
| PC5 | ≈ 7% | month, quarter, week-of-year | **Time of year** |
| PC6 | ≈ 5% | arrivals % change, arrivals vs 4w mean | **Supply shock** |
| PC7 | ≈ 4% | price vs state/national mean | **Local price premium/discount** |
| PC8 | ≈ 3% | heavy-rain flag, quarter-start flag (−) | **Extreme rain / quarter-turn events** |

In [ ]:
# 6c. Transform train + test; verify decorrelation
Z_train = pd.DataFrame(pipe.transform(train[NUMERIC_FEATURES]), columns=pc_names, index=train.index)
Z_test  = pd.DataFrame(pipe.transform(test[NUMERIC_FEATURES]),  columns=pc_names, index=test.index)

off = lambda m: np.abs(m.values[~np.eye(len(m), dtype=bool)]).max()
print(f"Max |corr| between PCs — train : {off(Z_train.corr()):.3f}  (uncorrelated by construction)")
print(f"Max |corr| between PCs — test  : {off(Z_test.corr()):.3f}  (distribution shift — monsoon Q3)")

rho_pc = Z_train.corrwith(train["target_pct_change"], method="spearman")
print("\nSpearman ρ of each PC with next-week price change (train):")
rho_pc.round(3)

**Caveat — test set distribution shift:** The test quarter (Q3, Jul–Sep) is the monsoon season, which is almost entirely absent from the training data (only one Q3 week in train). Season-related features (month, rainfall, humidity) move together in Q3, causing PC4, PC5 to become correlated on the test set even though they are uncorrelated on train. This is distribution shift, not a PCA bug — see Task 5 for mitigation.

In [ ]:
# Chart 11: Scatter of top-2 predictive PCs, coloured by target direction
top2   = rho_pc.abs().sort_values(ascending=False).index[:2]
sample = Z_train.join(train["target_direction"]).sample(min(3000, len(Z_train)), random_state=42)

fig, ax = plt.subplots(figsize=(8, 6))
for d, col in DIR_COLORS.items():
    s = sample[sample["target_direction"] == d]
    ax.scatter(s[top2[0]], s[top2[1]], s=10, alpha=0.45, color=col, label=d, edgecolors="none")
ax.set_xlabel(f"{top2[0]} (ρ with target = {rho_pc[top2[0]]:+.2f})")
ax.set_ylabel(f"{top2[1]} (ρ with target = {rho_pc[top2[1]]:+.2f})")
ax.legend(title="Next week", frameon=False)
ax.set_title("Chart 11: Training weeks on the two most predictive PCs")
fig.savefig(FIG / "11_pca_top2_components_by_direction.png")
plt.show()

## 7. Save outputs

In [ ]:
# Full feature table
out = feats[KEYS + NUMERIC_FEATURES + TARGETS + ["split"]]
out.to_csv(DATA / "features_panel.csv", index=False)
print(f"Saved data/features_panel.csv   {out.shape}")

# PCA feature table
pca_table = pd.concat(
    [feats[KEYS + TARGETS + ["split"]],
     pd.concat([Z_train, Z_test]).loc[feats.index].round(6)],
    axis=1,
)
pca_table.to_csv(DATA / "features_pca.csv", index=False)
print(f"Saved data/features_pca.csv      {pca_table.shape}")

# PCA pipeline
joblib.dump(pipe, MODELS / "pca_pipeline.joblib")
print("Saved models/pca_pipeline.joblib")

# Loadings + VIF reports
loadings.round(4).to_csv(REP / "pca_loadings.csv")
print("Saved reports/pca_loadings.csv")
print("Saved reports/feature_vif.csv")

## 8. Summary of findings

| # | Topic | Finding |
|---|---|---|
| 1 | Price dynamics | Tomato is dearest; all crops show a strong **quarterly cycle** (+6% from Q-start to Q-end). |
| 2 | Arrivals | Within-mandi arrivals are **uncorrelated** with price levels (cross-mandi levels dominate). Short-term supply deviation features are still included. |
| 3 | Weather | Unconditional weather categories have **almost no effect** on next-week price change; lagged and cumulative features retained for the model to weight. |
| 4 | Price change dist. | Weekly changes centre near 0%, spread ≈ ±6.5%. A ±2.5% flat band gives a balanced 38/24/38 class split. |
| 5 | Leakage | **Zero leakage** — truncation test passed at 3 cut-offs; max |Spearman ρ| with target = 0.43 (threshold 0.80). |
| 6 | Multicollinearity | 23/38 features have VIF > 10 (price-level group VIF up to ~6,700). PCA is necessary before linear modelling. |
| 7 | PCA | **14 components** retain **91%** of training variance and are perfectly uncorrelated on train. PC3 (quarterly cycle) and PC2 (momentum) are most predictive (|ρ| ≈ 0.37 and 0.26). |
| 8 | Distribution shift | Test quarter (Q3, monsoon) is nearly absent from training; season-related PCs become correlated on test. Task 5 should use rolling-origin CV. |

**Next step (Task 4 — Model Development):** build naive baseline, logistic regression (on PCs) and tree-based model (on raw features) with chronological train/test split and rolling-origin CV.